In [ ]:
# Install required packages
%pip install anthropic python-dotenv python-dotenv[cli] pandas

In [1]:
# Load environment variables from .env (expects ANTHROPIC_API_KEY)
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# --- Single-turn request example ---
from anthropic import Anthropic

# Instantiate the client (reads ANTHROPIC_API_KEY from the environment)
client = Anthropic()
model = "claude-haiku-4-5-20251001"

# Send a single message and store the full response object
message = client.messages.create(
    model=model,
    max_tokens = 1000,
    messages = [
        {
            "role":"user",
            "content": "tell me about quantum physics and how easy it is"
        }
    ]
)

In [ ]:
# Extract the text from the first content block of the response
from anthropic.types import TextBlock
block = message.content[0]
assert isinstance(block, TextBlock)
block.text

In [ ]:
# --- Multi-turn conversation helpers ---
from anthropic import Anthropic
from anthropic.types import TextBlock

client = Anthropic()
# model = "claude-3-haiku-20240307"
model = "claude-haiku-4-5-20251001"

# Append a user turn to the conversation history
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# Append an assistant turn to the conversation history
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Send the full conversation history and return the assistant reply
def chat(messages, system=None):
    kwargs = dict(model=model, max_tokens=1000, messages=messages)
    if system:
        kwargs["system"] = system
    message = client.messages.create(**kwargs)
    block = message.content[0]
    assert isinstance(block, TextBlock)
    return block.text

In [ ]:
# Start a fresh conversation and send the first user message
messages = []
add_user_message(messages, "Define quantum computing in one sentence")
messages

In [ ]:
# Get the assistant reply and save it to the history so context is preserved
answer = chat(messages)
add_assistant_message(messages, answer)
messages

In [ ]:
# Add a follow-up user message to continue the conversation
add_user_message(messages, "Write another sentence")

In [ ]:
# Send the updated history to get the next reply
answer = chat(messages)

In [ ]:
# Display the assistant answer
answer

In [ ]:
# --- Interactive chat loop ---
# Resets history and chats until the user submits an empty input
messages = []

while True:
    user_input = input(">")
    if not user_input.strip():  # exit loop on empty/whitespace-only input
        break
    print("user: ", user_input)
    add_user_message(messages, user_input)

    system = """
    You are a first grade teacher, 
    use Socrates mayeutics for the students

    """

    answer = chat(messages, system = system)
    add_assistant_message(messages, answer)  # keep assistant reply in history
    print("---")
    print(answer)
    print("---")
